# 10 · Layer 1 — Hybrid High-Recall Extraction
Overlapping chunking → coreference → **union** of (token-NER ∪ gazetteer ∪ LLM) → pass-2 differential sweep. The premise is that a single LLM pass misses low-salience entities; this notebook shows each component and writes the result into the mentions/assertions tables that Layers 2–4 consume.

In [ ]:
# --- bootstrap: make the src package importable from any working dir ---
import sys
from pathlib import Path
p = Path.cwd().resolve()
while not (p / 'config' / '00_config.py').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
print('project root:', p)


In [ ]:
from src import chunking, gazetteers, coref, ner_ensemble, sweep
from src.settings import CFG, Paths
print('chunk target:', CFG.CHUNK_TOKENS, 'tokens, overlap', CFG.CHUNK_OVERLAP_RATIO)
doc = sorted(Paths.raw_notes.glob('*.txt'))[1]
text = doc.read_text()
chunks = chunking.chunk_document(doc.stem, 'CLM0000', text)
print('chunks:', len(chunks), '| coverage:', chunking.coverage_report(text, chunks))


In [ ]:
# deterministic gazetteers: structured codes are never left to the LLM
for h in gazetteers.scan_valid(text):
    print(f'  {h.label:16s} {h.text!r}')


In [ ]:
# token-level NER: reads every literal token, regardless of salience
backend = ner_ensemble.get_token_ner()
print('backend:', backend.name, '(GlinerBackend activates when gliner is installed)')
for c in backend.extract(text, 0)[:10]:
    print(f'  {c.label:18s} {c.text!r}')


In [ ]:
# the UNION + sweep, with provenance showing which extractor found each span
spans = ner_ensemble.extract_chunk(chunks[0], backend)
extra = sweep.sweep_chunk(chunks[0], spans)
allspans = ner_ensemble.union_spans([spans, extra])
for c in allspans[:14]:
    print(f'  {c.label:16s} {c.text!r:34s} found_by={sorted(c.extractors)}')
print('\nresidual unmapped after sweep:', sweep.residual_report(text, allspans))


In [ ]:
# coreference: pronouns/descriptors are LINKS, never graph nodes
demo = 'Dr. Ruiz treated the claimant. Ace Collision billed us. '\
       'The shop inflated parts and the physician disagreed.'
ms = [{'start': demo.index('Dr. Ruiz'), 'end': demo.index('Dr. Ruiz')+8,
       'text': 'Dr. Ruiz', 'label': 'medical_provider'},
      {'start': demo.index('Ace Collision'), 'end': demo.index('Ace Collision')+13,
       'text': 'Ace Collision', 'label': 'repair_shop'}]
for l in coref.get_resolver().resolve(demo, ms):
    print(f'  {l.kind:10s} {l.surface!r:16s} -> {l.antecedent_surface!r} ({l.antecedent_class})')


In [ ]:
# run Layer 1 over the whole corpus into mentions/assertions
from src.repository import Repository
from src import pipeline_v2
repo = Repository()
print(pipeline_v2.run(repo))
print('\nmention provenance:'); print(repo.table('mentions')['extractor'].value_counts())
repo.close()
